__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
import numpy as np
import pandas as pd
import re
import string
from utils.constants import chat_words
import os

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

#### Preprocessing

In [ ]:
# DATAPATH = 'data/spam.csv'

# dataset_v1 = pd.read_csv(
#     DATAPATH,
#     header=0,
#     usecols=[0, 1],
#     names=["label", "message"],
#     encoding="utf-8",
#     encoding_errors="ignore"
# )

# df = dataset_v1.copy()
# df = df.dropna(subset=["label", "message"]).copy()
# df["label"] = df["label"].astype(str).str.strip().str.lower()
# df["message"] = df["message"].astype(str)

# print(f"Messages: {len(df):,}")
# print("Label counts:", df["label"].value_counts().to_dict())
# df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
1041,ham,I'm in class. Will holla later
5488,ham,K. I will sent it again
1365,spam,HOT LIVE FANTASIES call now 08707509020 Just 2...
2516,ham,Yes.i'm in office da:)
1355,ham,Convey my regards to him


In [3]:
def remove_number_middle_of_text(text):
    pattern = re.compile(r'[a-zA-Z]+')
    text = pattern.sub(r"", text)
    return " ".join(text)

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words:
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
        
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text

In [ ]:
# # text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
# # https://www.helo_texts.com
# # http://localhost:5000"""+df['message'][0])
# df['tokens'] = df['message'].apply(text_preprocessing)
# corpus_tokens = df['tokens'].tolist()

#### Remove the empty tokens

In [ ]:
# null_indices = df[df['tokens'].apply(lambda x: len(x) == 0)].index
# df.drop(null_indices, inplace=True)

In [ ]:
# ' '.join(df['tokens'].iloc[0])

'go juror point crazy available bugi n great world la e buffet cine get amores wat'

#### For deleting the memory using gc

In [6]:
# import gc

# # del X_train
# gc.collect()

In [ ]:
# df['cleaned_text'] = df['tokens'].apply(' '.join)
# DATA_DIR = os.path.abspath("./data")
# df.to_csv(f"{DATA_DIR}/processed.csv", index=True)

#### Processed data

Need to fix the cleaned text

In [4]:
df = pd.read_csv(os.path.abspath("./data/processed.csv"))
df.sample(5)


,Unnamed: 0,label,message,tokens,cleaned_text
4313,4320,ham,"K, jason says he's gonna be around so I'll be ...","['k', 'jason', 'say', 'hest', 'gona', 'around'...",k jason say hest gona around ill around lilt
3028,3033,ham,Get ready for &lt;#&gt; inches of pleasure...,"['get', 'ready', 'lilt', 'inch', 'pleasure']",get ready lilt inch pleasure
760,762,spam,Urgent Ur 500 guaranteed award is still unclai...,"['urgent', 'ur', '500', 'guarantee', 'award', ...",urgent ur 500 guarantee award still unclaimed ...
5027,5037,ham,Thanks for being there for me just to talk to ...,"['thank', 'talk', 'saturday', 'dear', 'cherish...",thank talk saturday dear cherish handhaving br...
1460,1463,ham,Ok good then i later come find _... C lucky i ...,"['ok', 'good', 'later', 'come', 'find', 'c', '...",ok good later come find c lucky tell go earl l...


### Features and targets

In [5]:
X = df['cleaned_text']
y = df['label']

#### Train and Test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (4449,)
y_train: (4449,)
X_test: (1113,)
y_test: (1113,)


#### Convert target to numerical

In [7]:
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

## Train

#### OneHotEncoder

In [91]:
all_train_tokens = list({
    token 
    for line in X_train.tolist() 
    if isinstance(line, str) 
    for token in line.split() 
    if token and len(token) > 0 and token not in (0, 0.0, "0", "0.0")
})

In [93]:
len(all_train_tokens)
all_train_tokens_arr = np.array(all_train_tokens).reshape(-1, 1)

In [94]:
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.int8)
ohe.fit(all_train_tokens_arr)

vocab_size = len(ohe.categories_[0])

def doc_to_ohe(corpus, ohe_encoder=ohe):
    ohe_corpus = []
    for line in corpus.tolist():
        # 1. Ensure line is a string to avoid AttributeError on floats/NaNs
        if isinstance(line, str):
            line_tokens = [
                tok for tok in line.split() 
                if tok and tok not in (0, 0.0, "0", "0.0")
            ]
            
            # 2. Only transform if there are valid tokens left
            if line_tokens:
                ohe_line = ohe_encoder.transform(np.array(line_tokens).reshape(-1, 1)).toarray()
                ohe_corpus.append(ohe_line)
                
    return ohe_corpus


In [95]:
X_train_encoded = doc_to_ohe(X_train)

encoded_max_length = max(len(line) for line in X_train_encoded)
print(encoded_max_length)


79


In [96]:
X_train_ohe = []

for ohe_line in X_train_encoded:
    # Pad each line to the maximum length with zeros
    padded_line = np.pad(ohe_line, ((0, encoded_max_length - len(ohe_line)), (0, 0)), mode='constant')
    X_train_ohe.append(padded_line)

In [97]:
X_train_ohe[0]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(79, 6233), dtype=int8)

#### Bag Of Words

In [98]:
train_corpus = X_train
test_corpus = X_test

cv = CountVectorizer()
cv_ngram = CountVectorizer(ngram_range=(1, 2))
X_train_bow = cv.fit_transform(train_corpus)
X_test_bow = cv.transform(test_corpus)

X_train_bow_ngram = cv_ngram.fit_transform(train_corpus)
X_test_bow_ngram = cv_ngram.transform(test_corpus)

In [ ]:
print(f'After apply the BoW(Bag of Word) the shape: {X_train_bow.shape}')
print(f'Vocabulary size: {len(cv.vocabulary_)}')
# cv.get_feature_names_out()[1000]

### TF-IDF

In [111]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [112]:
len(tfidf.vocabulary_)

6201

In [114]:
new_text = ["Hello World how are you!"]

tfidf.transform(new_text).toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 6201))

In [121]:
new_text = ["Once upon a time in BD."]

tfidf.transform(new_text).toarray()



array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 6213))

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'System device: {device}')

System device: cuda


### Word2Vec

In [ ]:
from gensim.models import KeyedVectors
import gensim.downloader as api
word2vec_model = api.load('word2vec-google-news-300')
# wv_model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True, limit=500000)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
def text_to_vector(texts, wv=word2vec_model, dim=300):
    vectors = []
    for text in texts:
        tokens = text.split()
        token_vectors = []
        for token in tokens:
            if token in wv:
                token_vectors.append(wv[token])
            else:

In [20]:
text_to_vector(X_train.tolist()[1])

['b']

#### Convert to `Torch`

In [13]:
word2vec_model['hello'].shape

(300,)

In [122]:

X_train = torch.tensor(X_train_tfidf, dtype=torch.long)
X_test = torch.tensor(X_test_tfidf, dtype=torch.long)


In [123]:
X_train.shape

torch.Size([4457, 6213])